<a href="https://colab.research.google.com/github/r3belli0us/week-2/blob/main/Week%202%20Challenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install google-play-scraper pandas transformers torch psycopg2-binary

In [ ]:
import pandas as pd
from google_play_scraper import Sort, reviews
from transformers import pipeline

app_ids = {
    'CBE': 'com.combanketh.mobilebanking',
    'BOA': 'com.boa.boaMobileBanking',
    'Dashen': 'com.dashen.dashensuperapp'
}

all_reviews = []

print("Scraping starting...")
for bank, app_id in app_ids.items():
    try:
        result, _ = reviews(
            app_id,
            lang='en',
            country='et',
            sort=Sort.NEWEST,
            count=400
        )
        for r in result:
            all_reviews.append({
                'review_id': r['reviewId'],
                'review_text': r['content'],
                'rating': r['score'],
                'review_date': r['at'],
                'bank_name': bank,
                'source': 'Google Play'
            })
    except Exception as e:
        print(f"Skipped {bank} due to fetch error: {e}")

df = pd.DataFrame(all_reviews)

df = df.drop_duplicates(subset=['review_id'])
df = df.dropna(subset=['review_text', 'rating'])
df['review_date'] = pd.to_datetime(df['review_date']).dt.strftime('%Y-%m-%d')

print("Running AI Sentiment Analysis...")
sentiment_pipeline = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

def get_sentiment(row):
    text = str(row['review_text']).strip()
    rating = row['rating']

    if len(text.split()) <= 2:
        if rating >= 4: return "POSITIVE", 1.0
        if rating <= 2: return "NEGATIVE", 1.0
        return "NEUTRAL", 1.0

    try:
        result = sentiment_pipeline(text[:512])[0]
        return result['label'], result['score']
    except:
        return "NEUTRAL", 0.0

df[['sentiment_label', 'sentiment_score']] = df.apply(
    lambda r: pd.Series(get_sentiment(r)), axis=1
)

df['identified_theme'] = 'General Feedback'

df.to_csv('cleaned_bank_reviews.csv', index=False)
print("Done! Data saved to cleaned_bank_reviews.csv")
display(df.head())

print(df['bank_name'].value_counts())

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore', category=FutureWarning)

df = pd.read_csv('cleaned_bank_reviews.csv')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sns.set_theme(style="whitegrid")

sns.countplot(
    data=df,
    x='bank_name',
    hue='sentiment_label',
    palette={'POSITIVE': 'green', 'NEGATIVE': 'red', 'NEUTRAL': 'blue'},
    ax=axes[0]
)
axes[0].set_title('Customer Sentiment Comparison', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Bank Name', fontsize=12)
axes[0].set_ylabel('Number of Reviews', fontsize=12)

sns.barplot(
    data=df,
    x='bank_name',
    y='rating',
    hue='bank_name',
    legend=False,
    errorbar=None,
    palette='Blues_d',
    ax=axes[1]
)
axes[1].set_title('Average Star Rating (Out of 5)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Bank Name', fontsize=12)
axes[1].set_ylabel('Average Rating', fontsize=12)
axes[1].set_ylim(0, 5)

plt.tight_layout()
plt.show()

plt.savefig('sentiment_charts.png', dpi=300)